# Fase 3: Evaluación de Impactos

## Congestión, Costo-Beneficio Social y Valorización del Suelo

### Componentes:
1. **Congestión**: vehículos-km ahorrados, horas-pico reducidas
2. **Costo-Beneficio**: monetización de ahorros vs inversión
3. **Valorización del suelo**: modelo hedónico de precios

In [ ]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
import sys, os, warnings
sys.path.insert(0, os.path.abspath('../src'))
warnings.filterwarnings('ignore')

from config import PROCESSED_DATA, FIGURES
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

trip_gen = pd.read_csv(PROCESSED_DATA / 'trip_generation.csv')
matrices = {}
for k in ['car', 'bus', 'metro_base', 'metro_full', 'train']:
    m = pd.read_csv(PROCESSED_DATA / f'tt_{k}.csv', index_col=0)
    matrices[k] = m.values.astype(float)

---
## 1. Demanda

In [ ]:
from demand_model import estimate_demand
r_full = estimate_demand(matrices, trip_gen, 'full')
r_base = estimate_demand(matrices, trip_gen, 'base')

---
## 2. Congestión

In [ ]:
from congestion import compute_congestion_impact

c_full = compute_congestion_impact(
    r_full['T_metro'], r_full['T_train'],
    matrices['car'], matrices['metro_full'], 'Red Completa')

c_base = compute_congestion_impact(
    r_base['T_metro'], r_base['T_train'],
    matrices['car'], matrices['metro_base'], 'Base')

---
## 3. Costo-Beneficio Social

In [ ]:
from cost_benefit import (
    compute_social_benefits, compute_investment_cost, compute_cost_benefit,
    LINE_LENGTHS
)

benefits = compute_social_benefits(c_full, 'Red Completa')
total_inv, inv_df = compute_investment_cost()
cb = compute_cost_benefit(benefits, total_inv, 'Red Completa')

---
## 4. Valorización del Suelo

In [ ]:
from land_value import simulate_land_values, plot_land_value

zones = gpd.read_file(str(PROCESSED_DATA / 'zones.gpkg'), layer='zones')
stations = gpd.read_file(str(PROCESSED_DATA / 'stations.gpkg'), layer='stations')

lv_results, total_gain = simulate_land_values(zones, stations)
print(f'\nPlusvalía total estimada: S/{total_gain:,.0f}')
print(f'\nTop 10 distritos por plusvalía:')
print(lv_results.sort_values('Plusvalía_S/', ascending=False).head(10).to_string(index=False))

In [ ]:
plot_land_value(lv_results, zones)
plt.show()

---
## 5. B/C por Línea

In [ ]:
line_results = []
for lid, km in LINE_LENGTHS.items():
    if lid in ['TREN_ICA', 'TREN_NORTE']:
        inv = km * 50_000_000 * 3.7
    else:
        inv = km * 150_000_000 * 3.7
    pax_share = km / sum(LINE_LENGTHS.values())
    line_benefits = benefits['npv_benefit_30yr'] * pax_share
    bc = line_benefits / inv if inv > 0 else 0
    line_results.append({'Línea': lid, 'km': km,
                         'Inversión (M S/)': f'{inv/1e6:.0f}',
                         'B/C': round(bc, 2)})

bc_df = pd.DataFrame(line_results)
print(bc_df.to_string(index=False))

---
## Resumen Fase 3

| Indicador | Valor |
|-----------|-------|
| Viajes transferidos a TP/día | ... |
| Horas ahorradas/día | ... |
| Vehículos retirados | ... |
| B/C promedio (red completa) | ... |
| Plusvalía total estimada | ... |

**Siguiente paso:** Fase 4 — Benchmarking internacional